In [48]:
import json
import os

In [49]:
data = []
with open("../datasets/processed/opencode_sft_filtered.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        data.append({
            "prompt": record.get("prompt"),
            "response": record.get("response"),
            "solution": record.get("solution"),
            "difficulty": record.get("difficulty"),
            "source": record.get("source")
        })

In [50]:
#convert to df
import pandas as pd
df = pd.DataFrame(data, columns=["prompt", "response","solution", "difficulty", "source"])
df.head()

,prompt,response,solution,difficulty,source
0,Problem description.\nVipul is a hardworking s...,"<think>\nOkay, I need to solve this problem wh...",T = int(input())\nfor _ in range(T):\n s = ...,UNKNOWN_DIFFICULTY,codechef
1,The Chef likes to stay in touch with his staff...,"<think>\nOkay, let's see. The problem is to de...",import sys\n\ndef main():\n T = int(sys.std...,1,codechef
2,Frank explained its friend Felman the algorith...,"<think>\nOkay, let's see. I need to compute th...",import math\n\nn = int(input())\nfor _ in rang...,MEDIUM,codechef
3,An established group of scientists are working...,"<think>\nOkay, I need to solve this subset sum...",import sys\n\ndef main():\n input = sys.std...,UNKNOWN_DIFFICULTY,codechef
4,Let's call a string a phone number if it has l...,"<think>\nOkay, let's see. The problem is about...",n = int(input())\ns = input().strip()\ncount_8...,7,codeforces


In [51]:
df['difficulty'].unique()

<ArrowStringArray>
['UNKNOWN_DIFFICULTY',                  '1',             'MEDIUM',
                  '7',        'MEDIUM_HARD',               'EASY',
               'HARD',          'interview',        'competition',
                  '0',                  '2',                  '8',
                  '9',                 '10',          'VERY_HARD',
                 '11',                  '6',       'introductory',
                 '12',                  '3',                 '13',
                 '20',                 '14',                 '15',
                 '24',                 '17',                 '21',
                 '19']
Length: 28, dtype: str

In [52]:
df['source'].unique()

<ArrowStringArray>
['codechef', 'codeforces', 'hackerearth', 'atcoder', 'aizu']
Length: 5, dtype: str

In [53]:
len(df)

10000

In [54]:
word_counts = []

for _, row in df.iterrows():
    count = len(str(row['response']).split())
    word_counts.append(count)

# Statistics
print(f"Mean   : {pd.Series(word_counts).mean():.2f}")
print(f"Median : {pd.Series(word_counts).median():.2f}")
print(f"Min    : {pd.Series(word_counts).min()}")
print(f"Max    : {pd.Series(word_counts).max()}")
print(f"Std    : {pd.Series(word_counts).std():.2f}")

print("\nPercentiles")
print(f"25%    : {pd.Series(word_counts).quantile(0.25)}")
print(f"50%    : {pd.Series(word_counts).quantile(0.50)}")
print(f"75%    : {pd.Series(word_counts).quantile(0.75)}")
print(f"90%    : {pd.Series(word_counts).quantile(0.90)}")
print(f"95%    : {pd.Series(word_counts).quantile(0.95)}")
print(f"99%    : {pd.Series(word_counts).quantile(0.99)}")

Mean   : 2624.68
Median : 2493.50
Min    : 547
Max    : 5977
Std    : 1291.47

Percentiles
25%    : 1487.0
50%    : 2493.5
75%    : 3660.0
90%    : 4473.0
95%    : 4845.0
99%    : 5339.01


In [55]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
text = tokenizer.apply_chat_template(
    row['response'],
    tokenize=False,
    add_generation_prompt=True
)

model_inputs = tokenizer([text], return_tensors="pt").to('cuda')
model_inputs

{'input_ids': tensor([[151644,   8948,    198,   2610,    525,   1207,  16948,     11,   3465,
            553,  54364,  14817,     13,   1446,    525,    264,  10950,  17847,
             13, 151645,    198, 151644,  77091,    198]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
       device='cuda:0')}

In [56]:
model_inputs.input_ids.shape[1]

24

In [57]:
import pandas as pd
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name)

token_counts = []

for _, row in df.iterrows():
    count = len(
        tokenizer.encode(
            str(row["response"]),
            add_special_tokens=False
        )
    )
    token_counts.append(count)

stats = pd.Series(token_counts)

print(f"Mean   : {stats.mean():.2f}")
print(f"Median : {stats.median():.2f}")
print(f"Min    : {stats.min()}")
print(f"Max    : {stats.max()}")
print(f"Std    : {stats.std():.2f}")

print("\nPercentiles")
print(f"25%    : {stats.quantile(0.25)}")
print(f"50%    : {stats.quantile(0.50)}")
print(f"75%    : {stats.quantile(0.75)}")
print(f"90%    : {stats.quantile(0.90)}")
print(f"95%    : {stats.quantile(0.95)}")
print(f"99%    : {stats.quantile(0.99)}")

Mean   : 4276.53
Median : 4112.00
Min    : 1024
Max    : 8192
Std    : 2062.76

Percentiles
25%    : 2444.75
50%    : 4112.0
75%    : 5986.0
90%    : 7282.200000000001
95%    : 7693.0999999999985
99%    : 8110.01
